# Versión 2 · Descarga de datos
## Nubosidad, precipitación y desempeño de las plantas solares fotovoltaicas de Colombia

**Idea:** el factor de planta (energía real ÷ energía máxima posible) de las plantas solares de Colombia cambia **entre regiones** (Caribe, Magdalena Medio, Llanos, valles interandinos) y **entre meses** (temporadas secas y lluviosas). La hipótesis es que esas diferencias se explican sobre todo por la **nubosidad** y la **precipitación**. El objetivo es medir cuánto, y si esa relación cambia en el espacio.

**Unidad de análisis:** cada planta solar que reporta a XM, mes a mes. El resultado de este notebook es una **tabla panel planta × mes** lista para correlaciones, regresiones y estadística espacial.

| Tipo | Dato | Fuente | Resolución | Cuenta |
|---|---|---|---|---|
| Punto | Generación real por planta solar → factor de planta mensual | XM (`pydataxm`) | horaria → mensual | No |
| Punto | Ubicación de las plantas | Tabla base (Global Energy Monitor) + OpenStreetMap (automático) + revisión manual | — | No |
| Punto | Nubosidad, radiación, índice de claridad y lluvia en cada planta | NASA POWER (API) | mensual, ~0,5° | No |
| Punto | **Estaciones del IDEAM:** precipitación, temperatura y humedad observadas | IDEAM, Datos Abiertos (datos.gov.co) | 10 min → diaria → mensual | No |
| Punto | **Precipitación diaria CHIRPS en cada planta** → lluvia y días de lluvia por mes | UCSB/CHC vía ERDDAP de NOAA | diaria, 0,05° | No |
| Área | Departamentos y municipios | geoBoundaries | — | No |
| Área | Régimen de lluvias (unimodal/bimodal) | IDEAM, OGC `Clima_Analisis_Climatico` (32) | — | No |
| Área | Zonas de radiación solar promedio multianual | IDEAM, OGC `Clima_Potencial_Energetico` (25) | — | No |
| Superficie | Precipitación mensual CHIRPS v2.0 | UCSB/CHC vía ERDDAP de NOAA | 0,05° (~5 km) | No |
| Superficie | Frecuencia de nubes MODIS (climatología mensual) | EarthEnv (Wilson y Jetz, 2016) | ~1 km | No |
| Superficie | Elevación en cada planta | Copernicus GLO-30 (AWS) | 30 m | No |

**No necesitas ninguna cuenta.** Todo cabe en **< 1 GB** y tarda **< 30 min**.

## 0. Preparación

In [ ]:
import os, sys, subprocess
from pathlib import Path
PAQUETES = ["numpy", "pandas", "requests", "xarray", "netCDF4", "geopandas", "rasterio", "shapely",
            "pydataxm", "matplotlib"]
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/solar_colombia_v2"); CARPETA.mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PAQUETES], check=True)
print("Carpeta de trabajo:", os.getcwd())

## 1. Parámetros

In [ ]:
from datetime import date

INICIO = date(2023, 1, 1)        # periodo de análisis (meses completos)
FIN = date(2024, 12, 31)
BBOX_COL = (-79.1, -4.3, -66.8, 13.5)          # lon_min, lat_min, lon_max, lat_max (Colombia continental)

# Tabla base de ubicaciones (Global Energy Monitor). 'clave' = palabra para encontrar la planta en XM.
# Coordenadas 'aprox' → verifícalas en Google Earth. Agrega filas si una planta no se ubica sola.
UBICACIONES_BASE = [
    dict(clave="FUNDACION",      nombre="Fundación (Enel)",            lat=10.4812, lon=-74.2263, dpto="Magdalena", precision="exacta GEM"),
    dict(clave="LA LOMA",        nombre="La Loma (Enel)",              lat=9.7063,  lon=-73.5222, dpto="Cesar",     precision="aprox GEM"),
    dict(clave="TEPUY",          nombre="Tepuy (EPM)",                 lat=5.4689,  lon=-74.6711, dpto="Caldas",    precision="aprox GEM"),
    dict(clave="BOSQUES SOLARES",nombre="Bosques Solares de los Llanos",lat=4.2180, lon=-72.0221, dpto="Meta",      precision="exacta GEM"),
    dict(clave="SINCE",          nombre="Sincé",                       lat=9.2473,  lon=-75.1488, dpto="Sucre",     precision="aprox GEM"),
    dict(clave="CAMPANO",        nombre="Campano",                     lat=9.1064,  lon=-75.3989, dpto="Córdoba",   precision="aprox GEM"),
    dict(clave="PLANETA RICA",   nombre="Planeta Rica",                lat=8.4416,  lon=-75.6025, dpto="Córdoba",   precision="exacta GEM"),
]

DESCARGAR = {"areas": True, "xm": True, "ubicaciones_osm": True, "power": True,
             "chirps": True, "nubes_earthenv": True, "elevacion": True,
             "ideam_estaciones": True, "chirps_diario": True}

RADIO_EST_KM = 30          # estaciones del IDEAM a menos de esta distancia de cada planta
UMBRAL_LLUVIA_MM = 1.0     # un día cuenta como lluvioso si llueve al menos esto
MIN_DIAS_MES = 20          # días con dato que debe tener una estación para aceptar su valor mensual

BASE = Path.cwd()
RAW, PROC, OUT = BASE / "datos" / "crudos", BASE / "datos" / "procesados", BASE / "resultados"
for p in (RAW, PROC, OUT):
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
import json, re, time, unicodedata, difflib, urllib.request
import numpy as np
import pandas as pd
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

def get(url, **kw):
    for i in range(4):
        try:
            r = requests.get(url, timeout=kw.pop("timeout", 300), **kw); r.raise_for_status(); return r
        except Exception as e:
            print(f"   reintento {i + 1}: {e}"); time.sleep(10)
    raise RuntimeError(url)

def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode().upper()
    t = re.sub(r"\b(PARQUE|PLANTA|GRANJA|SOLAR|FOTOVOLTAIC[OA]|PSF|PV|SAS|S\.A\.S|ESP|DE|DEL|LA|EL|LOS)\b", " ", t)
    return re.sub(r"[^A-Z0-9 ]", " ", re.sub(r"\s+", " ", t)).strip()

def meses(a, b):
    return pd.period_range(pd.Timestamp(a), pd.Timestamp(b), freq="M")

## 2. Datos tipo ÁREA
### 2.1 Departamentos y municipios de Colombia (geoBoundaries)

In [ ]:
def geoboundaries(nivel):
    with urllib.request.urlopen(f"https://www.geoboundaries.org/api/current/gbOpen/COL/{nivel}/", timeout=120) as r:
        return gpd.read_file(json.load(r)["gjDownloadURL"]).to_crs(4326)

for nivel, nombre in (("ADM1", "departamentos"), ("ADM2", "municipios")):
    f = PROC / f"{nombre}.gpkg"
    if DESCARGAR["areas"] and not f.exists():
        geoboundaries(nivel).to_file(f, driver="GPKG")
dptos = gpd.read_file(PROC / "departamentos.gpkg")
mpios = gpd.read_file(PROC / "municipios.gpkg")
print(f"{len(dptos)} departamentos · {len(mpios)} municipios")

### 2.2 Capas climáticas del IDEAM (servicio OGC API Features)
- **Régimen pluviométrico 1981–2010** (campo `regimen`): agrupa las plantas en regiones con **una** o **dos** temporadas de lluvia. Sirve para comparar la estacionalidad del factor de planta.
- **Radiación solar global promedio multianual** (campo `rango`, kWh/m²·día): el recurso solar esperado de cada zona.

In [ ]:
def ogc_ideam(servicio, coleccion, bbox, lote=1000):
    url = (f"https://visualizador.ideam.gov.co/gisserver/rest/services/{servicio}"
           f"/OGCFeatureServer/collections/{coleccion}/items")
    feats, offset = [], 0
    while True:
        fc = get(url, params={"f": "json", "bbox": ",".join(map(str, bbox)), "limit": lote, "offset": offset}).json()
        nuevos = fc.get("features", []); feats += nuevos
        if len(nuevos) < lote or not any(l.get("rel") == "next" for l in fc.get("links", [])):
            break
        offset += lote
    g = gpd.GeoDataFrame.from_features(feats, crs=4326)
    g["geometry"] = g.geometry.buffer(0)
    return g

capas_ideam = {"regimen_pluviometrico": ("Clima_Analisis_Climatico", 32),
               "zonas_radiacion": ("Clima_Potencial_Energetico", 25)}
for nombre, (serv, col) in capas_ideam.items():
    f = PROC / f"{nombre}_ideam.gpkg"
    if DESCARGAR["areas"] and not f.exists():
        print("Descargando", nombre, "…")
        ogc_ideam(serv, col, BBOX_COL).to_file(f, driver="GPKG")
regimen = gpd.read_file(PROC / "regimen_pluviometrico_ideam.gpkg")
zrad = gpd.read_file(PROC / "zonas_radiacion_ideam.gpkg")
lim = zrad["rango"].str.extract(r"([\d.]+)\s*-\s*([\d.]+)").astype(float)
zrad["rad_kwh_m2_dia"] = lim.mean(axis=1)
print("Regímenes:", regimen["regimen"].value_counts().to_dict())
print("Rangos de radiación:", sorted(zrad["rango"].dropna().unique()))

## 3. Datos tipo PUNTO
### 3.1 Plantas solares en XM y su generación mensual
1. Listado de recursos de XM → se guardan los **solares**.
2. Generación horaria `Gene/Recurso`, mes a mes → **energía mensual (MWh)**.
3. Capacidad: se intenta la métrica de XM `CapEfecNeta`. Si no está disponible, se usa como respaldo la **máxima generación horaria observada** (`cap_proxy_mw`).
4. **Factor de planta mensual** = energía ÷ (capacidad × horas del mes).

In [ ]:
from pydataxm.pydataxm import ReadDB
api = ReadDB()

def col(df, *claves):
    for c in df.columns:
        if any(k in c.lower() for k in claves):
            return c
    raise KeyError(f"{claves} no está en {list(df.columns)}")

f_rec = PROC / "xm_recursos_solares.csv"
if DESCARGAR["xm"] and not f_rec.exists():
    rec = api.request_data("ListadoRecursos", "Sistema", FIN, FIN)
    rec.to_csv(RAW / "xm_listado_recursos.csv", index=False)
    rec[rec.astype(str).apply(lambda f: " ".join(f).upper(), axis=1).str.contains("SOLAR")].to_csv(f_rec, index=False)
rec_sol = pd.read_csv(f_rec, dtype=str)
C_COD, C_NOM = col(rec_sol, "code", "codigo"), col(rec_sol, "name", "nombre")
CODIGOS = set(rec_sol[C_COD])
print(f"{len(rec_sol)} recursos solares en XM")
rec_sol[[C_COD, C_NOM]].head(40)

In [ ]:
# Consulta DIRECTA a la API de XM (sin pydataxm): evita el error de asyncio en Jupyter / Python ≥ 3.12
XM_URL = "https://servapibi.xm.com.co/{}"

def xm_api(metrica, entidad, inicio, fin, endpoint="hourly", tramo=30):
    filas, a, fin = [], pd.Timestamp(inicio), pd.Timestamp(fin)
    while a <= fin:
        b = min(a + pd.Timedelta(days=tramo - 1), fin)
        cuerpo = {"MetricId": metrica, "StartDate": f"{a:%Y-%m-%d}", "EndDate": f"{b:%Y-%m-%d}",
                  "Entity": entidad, "Filter": []}
        for intento in range(4):
            try:
                r = requests.post(XM_URL.format(endpoint), json=cuerpo, timeout=300)
                r.raise_for_status()
                break
            except Exception as e:
                print(f"   reintento {intento + 1}: {e}"); time.sleep(10)
        else:
            raise RuntimeError(f"XM no respondió: {metrica} {a:%Y-%m-%d}")
        for item in r.json().get("Items", []):
            for clave, entidades in item.items():
                if clave.endswith("Entities"):                 # HourlyEntities, DailyEntities…
                    for ent in entidades:
                        filas.append({"Date": item.get("Date"), **ent.get("Values", {})})
        a = b + pd.Timedelta(days=1)
    return pd.DataFrame(filas)

f_gen = PROC / "xm_generacion_mensual.csv"
if DESCARGAR["xm"] and not f_gen.exists():
    partes = []
    for m in meses(INICIO, FIN):
        f_m = RAW / "xm" / f"gene_{m}.csv"; f_m.parent.mkdir(parents=True, exist_ok=True)
        if not f_m.exists():
            print("XM", m, "…")
            g = xm_api("Gene", "Recurso", m.start_time.date(), m.end_time.date())
            if g.empty:
                print("   sin datos"); pd.DataFrame(columns=["Date", "code"]).to_csv(f_m, index=False)
            else:
                g[g["code"].astype(str).isin(CODIGOS)].to_csv(f_m, index=False)
        partes.append(pd.read_csv(f_m, dtype={"code": str}))
    g = pd.concat(partes, ignore_index=True)
    horas = [c for c in g.columns if c.lower().startswith("hour")]
    g[horas] = g[horas].apply(pd.to_numeric, errors="coerce") / 1000           # kWh → MWh
    g["mes"] = pd.to_datetime(g["Date"]).dt.to_period("M")
    g["mwh_dia"] = g[horas].sum(axis=1, min_count=1)
    g["max_hora_mwh"] = g[horas].max(axis=1)
    mensual = g.groupby(["code", "mes"]).agg(energia_mwh=("mwh_dia", "sum"), dias_con_dato=("mwh_dia", "count"),
                                             max_hora_mwh=("max_hora_mwh", "max")).reset_index()
    mensual.rename(columns={"code": "codigo_xm"}).to_csv(f_gen, index=False)
mensual = pd.read_csv(f_gen, dtype={"codigo_xm": str})
mensual["mes"] = pd.PeriodIndex(mensual["mes"], freq="M")

# Capacidad: métrica diaria de XM (CapEfecNeta) si existe; si no, proxy = máxima generación horaria observada
cap = mensual.groupby("codigo_xm")["max_hora_mwh"].max().rename("cap_proxy_mw").to_frame()
try:
    ce = xm_api("CapEfecNeta", "Recurso", pd.Timestamp(FIN) - pd.Timedelta(days=6), FIN, endpoint="daily")
    ce = ce[ce["code"].astype(str).isin(CODIGOS)]
    val = next(c for c in ce.columns if c.lower() == "value")
    cruda = ce.assign(v=pd.to_numeric(ce[val], errors="coerce")).groupby("code")["v"].last()
    cap = cap.join(cruda.rename("cap_xm_mw"))
    if (cap["cap_xm_mw"] / cap["cap_proxy_mw"]).median() > 100:   # venía en kW → MW
        cap["cap_xm_mw"] /= 1000
except Exception as e:
    print("CapEfecNeta no disponible:", e)
if "cap_xm_mw" in cap:
    razon = cap["cap_xm_mw"] / cap["cap_proxy_mw"]
    dudosa = ~razon.between(0.8, 2.5)          # valores inconsistentes → se usa el proxy
    print(f"Capacidad XM usada en {(~dudosa).sum()} plantas; proxy en {dudosa.sum()} (revisa xm_capacidad.csv)")
    cap["capacidad_mw"] = cap["cap_xm_mw"].where(~dudosa, cap["cap_proxy_mw"])
else:
    cap["capacidad_mw"] = cap["cap_proxy_mw"]
cap.to_csv(PROC / "xm_capacidad.csv")
mensual = mensual.merge(cap[["capacidad_mw"]], left_on="codigo_xm", right_index=True)
mensual["horas_mes"] = mensual["mes"].dt.days_in_month * 24
mensual["factor_planta"] = mensual["energia_mwh"] / (mensual["capacidad_mw"] * mensual["horas_mes"])
mensual.loc[mensual["dias_con_dato"] < mensual["mes"].dt.days_in_month - 2, "factor_planta"] = np.nan   # meses incompletos
print(f"{mensual.codigo_xm.nunique()} plantas con generación · {len(mensual)} planta-mes")
mensual.groupby("codigo_xm")["factor_planta"].describe().round(3)

### 3.2 Ubicación de las plantas
Orden de prioridad:
1. **Tabla base** (`UBICACIONES_BASE`).
2. **OpenStreetMap** (consulta automática a Overpass: plantas solares mapeadas en Colombia), emparejadas por nombre.
3. **Revisión manual:** se genera `plantas_ubicacion_revisar.csv`. Completa ahí `lat` y `lon` de las plantas sin ubicación (búscalas en gem.wiki o Google Earth), guárdalo como `plantas_ubicacion.csv` y vuelve a ejecutar la celda.

In [ ]:
f_osm = RAW / "osm_plantas_solares.json"
SERVIDORES_OVERPASS = ["https://overpass-api.de/api/interpreter",
                       "https://overpass.kumi.systems/api/interpreter",
                       "https://overpass.private.coffee/api/interpreter"]
CABECERAS = {"User-Agent": "UNAL-AnalisisGeoespacial/1.0 (proyecto academico; plantas solares Colombia)",
             "Accept": "application/json"}
if DESCARGAR["ubicaciones_osm"] and not f_osm.exists():
    q = """[out:json][timeout:180];
    area["ISO3166-1"="CO"][admin_level=2]->.co;
    (nwr["power"="plant"]["plant:source"="solar"](area.co);
     nwr["power"="plant"]["generator:source"="solar"](area.co););
    out center tags;"""
    for servidor in SERVIDORES_OVERPASS:
        try:
            r = requests.post(servidor, data={"data": q}, headers=CABECERAS, timeout=240)
            r.raise_for_status()
            r.json()                                   # verifica que sea JSON válido
            f_osm.write_text(r.text, encoding="utf-8")
            print("OSM descargado de", servidor)
            break
        except Exception as e:
            print(f"   {servidor} falló: {e}")
            time.sleep(5)
    else:
        print("⚠️ Ningún servidor de OpenStreetMap respondió: se sigue solo con la tabla base y tu archivo manual.")
osm = pd.DataFrame([{"nombre_osm": e.get("tags", {}).get("name", ""),
                     "lat": e.get("lat", e.get("center", {}).get("lat")),
                     "lon": e.get("lon", e.get("center", {}).get("lon")),
                     "potencia_osm": e.get("tags", {}).get("plant:output:electricity", "")}
                    for e in json.loads(f_osm.read_text(encoding="utf-8"))["elements"]]) if f_osm.exists() else pd.DataFrame()
osm = osm[osm.nombre_osm.str.len() > 0] if len(osm) else osm
print(f"OSM: {len(osm)} plantas solares con nombre en Colombia")

f_manual = BASE / "plantas_ubicacion.csv"
filas = []
for _, r in rec_sol[rec_sol[C_COD].isin(mensual.codigo_xm.unique())].iterrows():
    nom, nn = r[C_NOM], normalizar(r[C_NOM])
    fila = {"codigo_xm": r[C_COD], "nombre_xm": nom, "lat": np.nan, "lon": np.nan, "fuente": "", "similitud": np.nan}
    base = [u for u in UBICACIONES_BASE if normalizar(u["clave"]) in nn]
    if base:
        fila.update(lat=base[0]["lat"], lon=base[0]["lon"], fuente=f"tabla base ({base[0]['precision']})", similitud=1.0)
    elif len(osm):
        sims = osm.nombre_osm.map(lambda o: difflib.SequenceMatcher(None, nn, normalizar(o)).ratio())
        i = sims.idxmax()
        if sims[i] >= 0.6:
            fila.update(lat=osm.at[i, "lat"], lon=osm.at[i, "lon"], fuente=f"OSM: {osm.at[i, 'nombre_osm']}",
                        similitud=round(sims[i], 2))
    filas.append(fila)
ubic = pd.DataFrame(filas)
if f_manual.exists():                                           # tus correcciones manuales mandan
    man = pd.read_csv(f_manual, dtype={"codigo_xm": str}).dropna(subset=["lat", "lon"])
    ubic = ubic.set_index("codigo_xm"); ubic.update(man.set_index("codigo_xm")[["lat", "lon"]])
    ubic.loc[ubic.index.isin(man.codigo_xm), "fuente"] = "manual"; ubic = ubic.reset_index()
ubic.to_csv(BASE / "plantas_ubicacion_revisar.csv", index=False)
print(f"Ubicadas: {ubic.lat.notna().sum()} de {len(ubic)} → revisa plantas_ubicacion_revisar.csv")
ubic.sort_values("fuente")

In [ ]:
plantas = gpd.GeoDataFrame(ubic.dropna(subset=["lat", "lon"]),
                           geometry=gpd.points_from_xy(ubic.dropna(subset=["lat", "lon"]).lon,
                                                       ubic.dropna(subset=["lat", "lon"]).lat), crs=4326)
# Atributos de área por superposición (punto en polígono)
plantas = gpd.sjoin(plantas, dptos[["shapeName", "geometry"]].rename(columns={"shapeName": "departamento"}),
                    how="left", predicate="within").drop(columns="index_right")
plantas = gpd.sjoin(plantas, regimen[["regimen", "geometry"]], how="left", predicate="within").drop(columns="index_right")
plantas = gpd.sjoin(plantas, zrad[["rango", "rad_kwh_m2_dia", "geometry"]], how="left", predicate="within") \
             .drop(columns="index_right").drop_duplicates("codigo_xm")
plantas.to_file(PROC / "plantas_solares.gpkg", driver="GPKG")
plantas.drop(columns="geometry")

### 3.3 Nubosidad, radiación y lluvia mensual en cada planta (NASA POWER)
API pública, sin cuenta (comunidad `RE`, energías renovables). Variables:
| Parámetro | Qué es | Unidad |
|---|---|---|
| `CLOUD_AMT` | Cobertura de nubes | % |
| `ALLSKY_SFC_SW_DWN` | Radiación global en superficie (cielo real) | kWh/m²·día |
| `CLRSKY_SFC_SW_DWN` | Radiación en cielo despejado | kWh/m²·día |
| `ALLSKY_KT` | Índice de claridad | 0–1 |
| `PRECTOTCORR` | Precipitación | mm/día |
| `T2M` | Temperatura a 2 m | °C |

In [ ]:
PARAMS = ["CLOUD_AMT", "ALLSKY_SFC_SW_DWN", "CLRSKY_SFC_SW_DWN", "ALLSKY_KT", "PRECTOTCORR", "T2M"]
f_power = PROC / "power_mensual_plantas.csv"
if DESCARGAR["power"] and not f_power.exists():
    filas = []
    for _, p in plantas.iterrows():
        url = (f"https://power.larc.nasa.gov/api/temporal/monthly/point?parameters={','.join(PARAMS)}"
               f"&community=RE&longitude={p.lon:.4f}&latitude={p.lat:.4f}&start={INICIO.year}&end={FIN.year}&format=JSON")
        par = get(url).json()["properties"]["parameter"]
        for clave in par[PARAMS[0]]:
            if clave.endswith("13"):                 # el mes "13" es el promedio anual
                continue
            filas.append({"codigo_xm": p.codigo_xm, "mes": f"{clave[:4]}-{clave[4:]}",
                          **{k: par[k].get(clave) for k in PARAMS}})
        time.sleep(1)
    pw = pd.DataFrame(filas).replace(-999, np.nan)
    pw.to_csv(f_power, index=False)
pw = pd.read_csv(f_power, dtype={"codigo_xm": str})
pw.groupby("codigo_xm")[PARAMS].mean().round(2)

### 3.4 Estaciones del IDEAM: precipitación, temperatura y humedad observadas
Datos medidos en tierra, publicados por el IDEAM en **Datos Abiertos** (datos.gov.co, API Socrata, sin cuenta):

| Conjunto | Variable | Paso original | Se guarda como |
|---|---|---|---|
| `hp9r-jxuu` | Catálogo Nacional de Estaciones (ubicación, altitud, categoría, estado) | — | tabla de estaciones |
| `s54a-sgyg` | Precipitación (mm) | 10 min (automáticas) / diaria | suma diaria → lluvia mensual y **días de lluvia** |
| `sbwg-7ju4` | Temperatura del aire a 2 m (°C) | 10 min – horaria | media, máxima y mínima diaria → mensual |
| `uext-mhny` | Humedad relativa del aire a 2 m (%) | 10 min – horaria | media diaria → mensual |

- Solo se consultan las estaciones a menos de `RADIO_EST_KM` de alguna planta.
- La agregación diaria la hace el servidor (`$group` de SoQL), así que no se descargan millones de registros de 10 minutos.
- Un mes de una estación se acepta si tiene al menos `MIN_DIAS_MES` días con dato.

**Para qué sirven:** son la única medición directa en tierra. Permiten **validar** la lluvia de NASA POWER y CHIRPS, contar los **días de lluvia** reales cerca de cada planta, y aportar la **temperatura** (los paneles pierden eficiencia con el calor) y la **humedad** (asociada a nubosidad baja). Para agregar otra variable, añade su identificador de datos.gov.co a `VARIABLES_IDEAM`.

In [ ]:
SOCRATA = "https://www.datos.gov.co/resource/{}.json"
VARIABLES_IDEAM = {   # nombre: (identificador en datos.gov.co, agregaciones diarias)
    "precipitacion": ("s54a-sgyg", {"valor": "sum"}),
    "temperatura":   ("sbwg-7ju4", {"valor": "avg", "vmax": "max", "vmin": "min"}),
    "humedad":       ("uext-mhny", {"valor": "avg"}),
}

def socrata(ident, params, lote=50000):
    """Consulta paginada a la API de datos.gov.co."""
    filas, offset = [], 0
    while True:
        bloque = get(SOCRATA.format(ident), params=params | {"$limit": lote, "$offset": offset}, timeout=600).json()
        if isinstance(bloque, dict):                       # mensaje de error de la API
            raise RuntimeError(bloque)
        filas += bloque
        if len(bloque) < lote:
            return pd.DataFrame(filas)
        offset += lote

def distancia_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

# 1) Catálogo de estaciones y estaciones cercanas a cada planta
f_cat = RAW / "ideam_catalogo_estaciones.csv"
if DESCARGAR["ideam_estaciones"] and not f_cat.exists():
    socrata("hp9r-jxuu", {}).drop(columns=["ubicaci_n"], errors="ignore").to_csv(f_cat, index=False)
cat = pd.read_csv(f_cat, dtype={"codigo": str})
cat["latitud"], cat["longitud"] = pd.to_numeric(cat["latitud"], errors="coerce"), pd.to_numeric(cat["longitud"], errors="coerce")
cat = cat.dropna(subset=["latitud", "longitud"])

cerca = []
for _, p in plantas.iterrows():
    d = distancia_km(p.lat, p.lon, cat.latitud.values, cat.longitud.values)
    sel = cat.loc[d <= RADIO_EST_KM, ["codigo", "nombre", "categoria", "tecnologia", "estado", "altitud", "latitud", "longitud"]].copy()
    sel["distancia_km"] = d[d <= RADIO_EST_KM].round(1)
    cerca.append(sel.assign(codigo_xm=p.codigo_xm))
est_plantas = pd.concat(cerca, ignore_index=True).sort_values(["codigo_xm", "distancia_km"])
est_plantas.to_csv(PROC / "ideam_estaciones_por_planta.csv", index=False)
CODIGOS_EST = sorted(est_plantas.codigo.unique())
print(f"{len(cat)} estaciones en el catálogo · {len(CODIGOS_EST)} a menos de {RADIO_EST_KM} km de alguna planta")
est_plantas.groupby("codigo_xm").size().rename("estaciones_cercanas")

In [ ]:
# 2) Series diarias de cada variable (agregadas en el servidor)
def diario_ideam(ident, aggs, codigos, tramo=40):
    sel = ", ".join(f"{f}(valorobservado::number) as {k}" for k, f in aggs.items())
    partes = []
    for i in range(0, len(codigos), tramo):
        lista = ",".join(f"'{c}'" for c in codigos[i:i + tramo])
        params = {"$select": f"codigoestacion, date_trunc_ymd(fechaobservacion) as dia, {sel}, count(valorobservado) as n_obs",
                  "$where": (f"codigoestacion in ({lista}) AND "
                             f"fechaobservacion between '{INICIO:%Y-%m-%d}T00:00:00' and '{FIN:%Y-%m-%d}T23:59:59'"),
                  "$group": "codigoestacion, dia", "$order": "codigoestacion, dia"}
        try:
            partes.append(socrata(ident, params))
        except Exception as e:                     # respaldo: descarga cruda y agrega aquí
            print("   agregación en el servidor falló, descargo crudo:", str(e)[:120])
            crudo = socrata(ident, {"$select": "codigoestacion, fechaobservacion, valorobservado", "$where": params["$where"]})
            if len(crudo):
                crudo["dia"] = pd.to_datetime(crudo.fechaobservacion).dt.floor("D")
                crudo["v"] = pd.to_numeric(crudo.valorobservado, errors="coerce")
                g = crudo.groupby(["codigoestacion", "dia"])["v"]
                partes.append(pd.DataFrame({k: g.agg({"avg": "mean"}.get(f, f)) for k, f in aggs.items()} | {"n_obs": g.count()}).reset_index())
        time.sleep(1)
    df = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    if len(df):
        df["dia"] = pd.to_datetime(df["dia"])
        for k in list(aggs) + ["n_obs"]:
            df[k] = pd.to_numeric(df[k], errors="coerce")
    return df

diarios = {}
for nombre, (ident, aggs) in VARIABLES_IDEAM.items():
    f = RAW / f"ideam_{nombre}_diaria.csv"
    if DESCARGAR["ideam_estaciones"] and not f.exists():
        print("IDEAM", nombre, "…")
        diario_ideam(ident, aggs, CODIGOS_EST).to_csv(f, index=False)
    diarios[nombre] = pd.read_csv(f, dtype={"codigoestacion": str}, parse_dates=["dia"])
    print(f"   {nombre}: {diarios[nombre].codigoestacion.nunique() if len(diarios[nombre]) else 0} estaciones con datos, "
          f"{len(diarios[nombre])} días-estación")

In [ ]:
# 3) Valores mensuales por estación
def mensual_estacion(df, columnas):
    if df.empty:
        return pd.DataFrame(columns=["codigoestacion", "mes", "dias_con_dato"])
    df = df.assign(mes=df.dia.dt.to_period("M").astype(str))
    g = df.groupby(["codigoestacion", "mes"])
    out = pd.DataFrame({"dias_con_dato": g.size()})
    for nuevo, (col, f) in columnas.items():
        out[nuevo] = g[col].agg(f)
    out = out.reset_index()
    out.loc[out.dias_con_dato < MIN_DIAS_MES, list(columnas)] = np.nan
    return out

pp = diarios["precipitacion"].assign(lluvioso=lambda d: d.valor >= UMBRAL_LLUVIA_MM)
est_mes = (mensual_estacion(pp, {"ideam_ppt_mm": ("valor", "sum"), "ideam_dias_lluvia": ("lluvioso", "sum"),
                                 "ideam_ppt_max_diaria_mm": ("valor", "max")})
           .merge(mensual_estacion(diarios["temperatura"], {"ideam_temp_media_c": ("valor", "mean"),
                                   "ideam_temp_max_c": ("vmax", "mean"), "ideam_temp_min_c": ("vmin", "mean")})
                  .drop(columns="dias_con_dato"), on=["codigoestacion", "mes"], how="outer")
           .merge(mensual_estacion(diarios["humedad"], {"ideam_hr_media_pct": ("valor", "mean")})
                  .drop(columns="dias_con_dato"), on=["codigoestacion", "mes"], how="outer"))
est_mes.to_csv(PROC / "ideam_estaciones_mensual.csv", index=False)

# Capa de puntos (tipo punto) con el promedio de cada estación en el periodo
prom = est_mes.groupby("codigoestacion").mean(numeric_only=True).reset_index()
est_gdf = cat.merge(prom, left_on="codigo", right_on="codigoestacion")
est_gdf = gpd.GeoDataFrame(est_gdf, geometry=gpd.points_from_xy(est_gdf.longitud, est_gdf.latitud), crs=4326)
est_gdf.to_file(PROC / "ideam_estaciones.gpkg", driver="GPKG")
print(f"{est_gdf.codigo.nunique()} estaciones con datos mensuales")
est_gdf[["codigo", "nombre", "categoria", "ideam_ppt_mm", "ideam_dias_lluvia", "ideam_temp_media_c", "ideam_hr_media_pct"]].round(1)

## 4. Datos tipo SUPERFICIE
### 4.1 Precipitación mensual CHIRPS v2.0 (0,05°)
Recorte de Colombia vía el servidor ERDDAP de NOAA CoastWatch (sin cuenta). Si ERDDAP no tiene meses recientes, la celda usa como respaldo los GeoTIFF mensuales de UCSB.

In [ ]:
import xarray as xr
f_chirps = RAW / "chirps_mensual_colombia.nc"
lo0, la0, lo1, la1 = BBOX_COL
if DESCARGAR["chirps"] and not f_chirps.exists():
    url = ("https://coastwatch.pfeg.noaa.gov/erddap/griddap/chirps20GlobalMonthlyP05.nc?"
           f"precip[({INICIO:%Y-%m-01}T00:00:00Z):1:({FIN:%Y-%m-01}T00:00:00Z)]"
           f"[({la0}):1:({la1})][({lo0}):1:({lo1})]")
    try:
        f_chirps.write_bytes(get(url, timeout=900).content)
    except Exception as e:
        print("ERDDAP falló, uso los GeoTIFF de UCSB:", e)
        import rasterio
        from rasterio.windows import from_bounds
        capas = []
        for m in meses(INICIO, FIN):
            u = f"/vsigzip//vsicurl/https://data.chc.ucsb.edu/products/CHIRPS-2.0/global_monthly/tifs/chirps-v2.0.{m.year}.{m.month:02d}.tif.gz"
            with rasterio.open(u) as src:
                w = from_bounds(lo0, la0, lo1, la1, src.transform)
                a = src.read(1, window=w).astype("float32"); a[a < 0] = np.nan
                t = src.window_transform(w)
            capas.append(a)
        ny, nx = capas[0].shape
        lats = t.f + t.e * (np.arange(ny) + 0.5); lons = t.c + t.a * (np.arange(nx) + 0.5)
        xr.Dataset({"precip": (("time", "latitude", "longitude"), np.stack(capas))},
                   coords={"time": meses(INICIO, FIN).to_timestamp(), "latitude": lats, "longitude": lons}).to_netcdf(f_chirps)
chirps = xr.open_dataset(f_chirps)
print(chirps["precip"].shape, "· meses:", chirps.sizes["time"])

### 4.2 Precipitación diaria CHIRPS en cada planta: días de lluvia
La lluvia mensual no dice si llovió un día fuerte o veinte días seguidos; para la generación solar importa más lo segundo. Por eso se descarga la **serie diaria** de CHIRPS en el píxel de cada planta (ERDDAP de NOAA, conjunto `chirps20GlobalDailyP05`, solo un píxel por planta: archivos muy pequeños) y se calcula por mes:
- `chirps_dias_lluvia`: días con lluvia ≥ `UMBRAL_LLUVIA_MM`,
- `chirps_ppt_max_diaria_mm`: el día más lluvioso,
- `chirps_ppt_diaria_mm`: suma de los días (sirve para comparar con el CHIRPS mensual).

In [ ]:
import io
f_chd = RAW / "chirps_diario_plantas.csv"
URL_CHD = ("https://coastwatch.pfeg.noaa.gov/erddap/griddap/chirps20GlobalDailyP05.csv?"
           "precip%5B({t0}T00:00:00Z):1:({t1}T00:00:00Z)%5D%5B({lat:.3f})%5D%5B({lon:.3f})%5D")
if DESCARGAR["chirps_diario"] and not f_chd.exists():
    partes = []
    for _, p in plantas.iterrows():
        try:
            txt = get(URL_CHD.format(t0=f"{INICIO:%Y-%m-%d}", t1=f"{FIN:%Y-%m-%d}", lat=p.lat, lon=p.lon), timeout=300).text
            d = pd.read_csv(io.StringIO(txt), skiprows=[1])            # la 2.ª fila son las unidades
            partes.append(d.assign(codigo_xm=p.codigo_xm))
        except Exception as e:
            print("   CHIRPS diario falló para", p.codigo_xm, ":", e)
        time.sleep(1)
    pd.concat(partes, ignore_index=True).to_csv(f_chd, index=False)
chd = pd.read_csv(f_chd, dtype={"codigo_xm": str})
chd["precip"] = pd.to_numeric(chd["precip"], errors="coerce")
chd["mes"] = pd.to_datetime(chd["time"]).dt.tz_localize(None).dt.to_period("M").astype(str)
chd_mes = (chd.groupby(["codigo_xm", "mes"])["precip"]
              .agg(chirps_ppt_diaria_mm="sum", chirps_ppt_max_diaria_mm="max",
                   chirps_dias_lluvia=lambda v: int((v >= UMBRAL_LLUVIA_MM).sum()))
              .reset_index())
chd_mes.to_csv(PROC / "chirps_diario_mensual_plantas.csv", index=False)
chd_mes.groupby("codigo_xm")[["chirps_ppt_diaria_mm", "chirps_dias_lluvia"]].mean().round(1)

### 4.3 Frecuencia de nubes MODIS · EarthEnv (~1 km)
Climatología: porcentaje de días nublados de cada mes del año (2000–2014), más el promedio anual (Wilson y Jetz, 2016). Muestra **dónde** es más nublado con mucho detalle espacial. Se recorta a Colombia leyendo solo esa ventana del archivo remoto.

In [ ]:
import rasterio
from rasterio.windows import from_bounds
carp_nubes = RAW / "earthenv_nubes"; carp_nubes.mkdir(parents=True, exist_ok=True)
archivos_nubes = {f"{m:02d}": f"MODCF_monthlymean_{m:02d}.tif" for m in range(1, 13)} | {"anual": "MODCF_meanannual.tif"}
if DESCARGAR["nubes_earthenv"]:
    for clave, nombre in archivos_nubes.items():
        destino = carp_nubes / f"nubes_{clave}_colombia.tif"
        if destino.exists():
            continue
        with rasterio.open(f"/vsicurl/https://data.earthenv.org/cloud/{nombre}") as src:
            w = from_bounds(*BBOX_COL, src.transform)
            a = src.read(1, window=w)
            perfil = src.profile | {"height": a.shape[0], "width": a.shape[1], "transform": src.window_transform(w),
                                    "driver": "GTiff", "compress": "lzw"}
        with rasterio.open(destino, "w", **perfil) as dst:
            dst.write(a, 1)
        print("ok", destino.name)
with rasterio.open(carp_nubes / "nubes_anual_colombia.tif") as src:
    print("Valores crudos de ejemplo:", np.unique(src.read(1)[::200, ::200])[:10],
          "→ EarthEnv guarda % × 100 (divide entre 100)")

### 4.4 Elevación en cada planta (Copernicus GLO-30)
Se lee solo el píxel de cada planta en la tesela remota.

In [ ]:
URL_DEM = ("/vsicurl/https://copernicus-dem-30m.s3.amazonaws.com/Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM/"
           "Copernicus_DSM_COG_10_{ns}{lat:02d}_00_{ew}{lon:03d}_00_DEM.tif")
if DESCARGAR["elevacion"] and "elevacion_m" not in plantas.columns:
    elev = []
    for _, p in plantas.iterrows():
        la, lo = int(np.floor(p.lat)), int(np.floor(p.lon))
        u = URL_DEM.format(ns="N" if la >= 0 else "S", lat=abs(la), ew="W" if lo < 0 else "E", lon=abs(lo))
        try:
            with rasterio.open(u) as src:
                elev.append(float(next(src.sample([(p.lon, p.lat)]))[0]))
        except Exception:
            elev.append(np.nan)
    plantas["elevacion_m"] = elev
    plantas.to_file(PROC / "plantas_solares.gpkg", driver="GPKG")
plantas[["nombre_xm", "departamento", "regimen", "rango", "elevacion_m"]]

## 5. Tabla panel planta × mes (lista para el análisis)
Une todo en una fila por planta y mes:
- **Variable respuesta:** factor de planta.
- **Variables explicativas:**
  - nubosidad y radiación (POWER),
  - lluvia (POWER, CHIRPS mensual y diario en el píxel de la planta: total, días de lluvia, día máximo),
  - estaciones del IDEAM más cercanas con dato ese mes: lluvia, días de lluvia, temperatura y humedad (y su distancia),
  - frecuencia de nubes de EarthEnv del mismo mes del año,
  - régimen de lluvias, zona de radiación, departamento y elevación.

In [ ]:
panel = mensual.copy()
panel["mes"] = panel["mes"].astype(str)
panel = panel.merge(pw, on=["codigo_xm", "mes"], how="left")
panel = panel.merge(plantas.drop(columns="geometry")[["codigo_xm", "nombre_xm", "lat", "lon", "departamento",
                                                      "regimen", "rango", "rad_kwh_m2_dia", "elevacion_m"]],
                    on="codigo_xm", how="inner")
# CHIRPS en el píxel de cada planta
ch = chirps["precip"]
lat_n = "latitude" if "latitude" in ch.dims else "lat"; lon_n = "longitude" if "longitude" in ch.dims else "lon"
vals = []
for _, p in plantas.iterrows():
    s = ch.sel({lat_n: p.lat, lon_n: p.lon}, method="nearest").to_series()
    s.index = pd.PeriodIndex(s.index, freq="M").astype(str)
    vals.append(s.rename("chirps_mm_mes").to_frame().assign(codigo_xm=p.codigo_xm))
panel = panel.merge(pd.concat(vals).reset_index(names="mes"), on=["codigo_xm", "mes"], how="left")
# EarthEnv: frecuencia de nubes del mismo mes del año
nub = {}
for m in range(1, 13):
    with rasterio.open(RAW / "earthenv_nubes" / f"nubes_{m:02d}_colombia.tif") as src:
        nub[m] = [float(v[0]) / 100 for v in src.sample(list(zip(plantas.lon, plantas.lat)))]
nub = pd.DataFrame(nub, index=plantas.codigo_xm.values).stack().rename("nubes_earthenv_pct").reset_index()
nub.columns = ["codigo_xm", "mes_del_año", "nubes_earthenv_pct"]
panel["mes_del_año"] = pd.PeriodIndex(panel["mes"], freq="M").month
panel = panel.merge(nub, on=["codigo_xm", "mes_del_año"], how="left")
# CHIRPS diario: días de lluvia y día más lluvioso de cada mes
panel = panel.merge(chd_mes, on=["codigo_xm", "mes"], how="left")
# IDEAM: para cada planta y mes, la estación más cercana que tenga dato de cada variable
cerc = est_plantas[["codigo_xm", "codigo", "distancia_km"]].merge(
    est_mes, left_on="codigo", right_on="codigoestacion").sort_values("distancia_km")
for cols, sufijo in ((["ideam_ppt_mm", "ideam_dias_lluvia", "ideam_ppt_max_diaria_mm"], "ppt"),
                     (["ideam_temp_media_c", "ideam_temp_max_c", "ideam_temp_min_c"], "temp"),
                     (["ideam_hr_media_pct"], "hr")):
    sel = (cerc.dropna(subset=[cols[0]]).groupby(["codigo_xm", "mes"]).first().reset_index()
               [["codigo_xm", "mes", *cols, "codigo", "distancia_km"]]
               .rename(columns={"codigo": f"ideam_est_{sufijo}", "distancia_km": f"ideam_dist_{sufijo}_km"}))
    panel = panel.merge(sel, on=["codigo_xm", "mes"], how="left")
panel.to_csv(PROC / "panel_plantas_mes.csv", index=False)
print(f"Panel: {panel.codigo_xm.nunique()} plantas × {panel.mes.nunique()} meses = {len(panel)} filas")
panel.head()

## 6. Inventario y mapa de verificación

In [ ]:
for f in sorted(PROC.glob("*")):
    print(f"{f.name:45s} {f.stat().st_size / 1e6:8.2f} MB")

fig, axs = plt.subplots(1, 3, figsize=(21, 8))
regimen.plot(ax=axs[0], column="regimen", cmap="Set3", legend=True, legend_kwds={"fontsize": 7, "loc": "lower left"})
dptos.boundary.plot(ax=axs[0], color="0.4", linewidth=0.3)
plantas.plot(ax=axs[0], color="k", marker="^", markersize=60)
axs[0].set_title("Régimen pluviométrico (IDEAM) y plantas solares", loc="left", fontweight="bold")

with rasterio.open(RAW / "earthenv_nubes" / "nubes_anual_colombia.tif") as src:
    a = src.read(1).astype(float) / 100; b = src.bounds
im = axs[1].imshow(np.where(a > 0, a, np.nan), extent=[b.left, b.right, b.bottom, b.top], cmap="Blues")
plt.colorbar(im, ax=axs[1], shrink=0.6, label="% de días nublados")
dptos.boundary.plot(ax=axs[1], color="0.3", linewidth=0.3); plantas.plot(ax=axs[1], color="#E0A526", marker="^", markersize=60, edgecolor="k")
axs[1].set_title("Frecuencia anual de nubes (EarthEnv/MODIS)", loc="left", fontweight="bold")

im = chirps["precip"].sum("time").plot(ax=axs[2], cmap="YlGnBu", add_colorbar=False)
plt.colorbar(im, ax=axs[2], shrink=0.6, label="mm en el periodo")
dptos.boundary.plot(ax=axs[2], color="0.3", linewidth=0.3); plantas.plot(ax=axs[2], color="#E0A526", marker="^", markersize=60, edgecolor="k")
axs[2].set_title("Precipitación acumulada CHIRPS", loc="left", fontweight="bold")
for ax in axs:
    ax.set_xlim(BBOX_COL[0], BBOX_COL[2]); ax.set_ylim(BBOX_COL[1], BBOX_COL[3])
plt.tight_layout(); plt.savefig(OUT / "verificacion_datos_v2.png", dpi=150); plt.show()

In [ ]:
# Primera mirada (no es el análisis): factor de planta mensual vs. nubosidad POWER
fig, ax = plt.subplots(figsize=(8, 6))
for reg, d in panel.dropna(subset=["factor_planta", "CLOUD_AMT"]).groupby("regimen"):
    ax.scatter(d.CLOUD_AMT, d.factor_planta, s=14, alpha=0.6, label=reg)
ax.set_xlabel("Nubosidad NASA POWER (%)"); ax.set_ylabel("Factor de planta mensual")
ax.legend(fontsize=7); ax.set_title("Exploración inicial por régimen de lluvias", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Mapas para la presentación
Exporta **7 mapas**, uno o dos por tipo de dato, con el mismo estilo y la proporción exacta de los espacios de la diapositiva "Datos espaciales" (vertical, 4 × 5,8 pulgadas, 200 dpi).

| Archivo | Tipo | Contenido |
|---|---|---|
| `01_plantas_factor_planta.png` | Punto | Plantas: tamaño = capacidad, color = factor de planta medio |
| `02_nubosidad_plantas.png` | Punto | Nubosidad media NASA POWER en cada planta |
| `03_regimen_pluviometrico.png` | Área | Régimen de lluvias del IDEAM |
| `04_zonas_radiacion.png` | Área | Zonas de radiación solar del IDEAM |
| `05_precipitacion_chirps.png` | Superficie | Precipitación media anual CHIRPS |
| `06_nubes_modis.png` | Superficie | Frecuencia anual de nubes (EarthEnv/MODIS) |
| `07_estaciones_ideam.png` | Punto | Estaciones del IDEAM cerca de las plantas: lluvia media anual |

Quedan en `resultados/mapas_presentacion/`. **Súbelos al chat** y los pongo en la diapositiva.

In [ ]:
from rasterio.features import geometry_mask
MAPAS = OUT / "mapas_presentacion"; MAPAS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
lo0, la0, lo1, la1 = BBOX_COL
FIG = (4, 5.8)
colombia = dptos.dissolve()

def base(ax, relleno=False):
    if relleno:
        dptos.plot(ax=ax, color="#F2F2EF", edgecolor="none", zorder=0)
    dptos.boundary.plot(ax=ax, color="0.45", linewidth=0.3, zorder=2)
    ax.set_xlim(lo0, lo1); ax.set_ylim(la0, la1); ax.set_aspect("equal"); ax.set_axis_off()

def barra(fig, ax, m, etiqueta):
    cb = fig.colorbar(m, ax=ax, orientation="horizontal", fraction=0.04, pad=0.01, aspect=30)
    cb.set_label(etiqueta, fontsize=10); cb.ax.tick_params(labelsize=8)

def guardar(fig, nombre):
    fig.savefig(MAPAS / nombre, dpi=200, bbox_inches="tight", facecolor="white"); plt.show()

resumen = panel.groupby("codigo_xm").agg(fp=("factor_planta", "mean"), cap=("capacidad_mw", "first"),
                                         nubes=("CLOUD_AMT", "mean")).reset_index()
pl = plantas.merge(resumen, on="codigo_xm")

# 1 · Plantas y factor de planta
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
m = ax.scatter(pl.geometry.x, pl.geometry.y, s=15 + 1.5 * pl.cap, c=100 * pl.fp, cmap="YlOrRd",
               edgecolor="k", linewidth=0.5, zorder=3)
barra(fig, ax, m, "Factor de planta medio (%) · tamaño = capacidad"); guardar(fig, "01_plantas_factor_planta.png")

# 2 · Nubosidad en cada planta
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
m = ax.scatter(pl.geometry.x, pl.geometry.y, s=70, c=pl.nubes, cmap="Blues", edgecolor="k", linewidth=0.5, zorder=3)
barra(fig, ax, m, "Nubosidad media NASA POWER (%)"); guardar(fig, "02_nubosidad_plantas.png")

# 3 · Régimen pluviométrico
fig, ax = plt.subplots(figsize=FIG)
regimen.plot(ax=ax, column="regimen", cmap="Set3", legend=True,
             legend_kwds={"loc": "upper center", "bbox_to_anchor": (0.5, -0.01), "fontsize": 7, "frameon": False, "ncol": 1})
base(ax); plantas.plot(ax=ax, color="k", marker="^", markersize=18, zorder=3); guardar(fig, "03_regimen_pluviometrico.png")

# 4 · Zonas de radiación
fig, ax = plt.subplots(figsize=FIG)
zrad.plot(ax=ax, column="rad_kwh_m2_dia", cmap="YlOrBr", edgecolor="none")
base(ax); plantas.plot(ax=ax, color="k", marker="^", markersize=18, zorder=3)
sm = plt.cm.ScalarMappable(cmap="YlOrBr", norm=plt.Normalize(zrad.rad_kwh_m2_dia.min(), zrad.rad_kwh_m2_dia.max()))
barra(fig, ax, sm, "Radiación global (kWh/m²·día)"); guardar(fig, "04_zonas_radiacion.png")

# 5 · Precipitación media anual CHIRPS
anual = chirps["precip"].sum("time") / (chirps.sizes["time"] / 12)
fig, ax = plt.subplots(figsize=FIG)
m = anual.plot(ax=ax, cmap="YlGnBu", add_colorbar=False, vmin=0, vmax=6000)
base(ax); plantas.plot(ax=ax, color="#E0A526", marker="^", markersize=18, edgecolor="k", zorder=3)
barra(fig, ax, m, "Precipitación media anual (mm)"); guardar(fig, "05_precipitacion_chirps.png")

# 6 · Frecuencia anual de nubes (EarthEnv), recortada a Colombia
with rasterio.open(RAW / "earthenv_nubes" / "nubes_anual_colombia.tif") as src:
    a = src.read(1).astype(float) / 100
    fuera = geometry_mask(colombia.geometry, out_shape=a.shape, transform=src.transform)
    b = src.bounds
a[fuera | (a <= 0)] = np.nan
fig, ax = plt.subplots(figsize=FIG)
m = ax.imshow(a, extent=[b.left, b.right, b.bottom, b.top], cmap="Greys", vmin=20, vmax=100)
base(ax); plantas.plot(ax=ax, color="#E0A526", marker="^", markersize=18, edgecolor="k", zorder=3)
barra(fig, ax, m, "Días nublados al año (%)"); guardar(fig, "06_nubes_modis.png")
print("Mapas en", MAPAS)

In [ ]:
# 7 · Estaciones del IDEAM cerca de las plantas (lluvia media anual observada)
e = est_gdf.dropna(subset=["ideam_ppt_mm"]).copy()
e["ppt_anual"] = e["ideam_ppt_mm"] * 12
fig, ax = plt.subplots(figsize=FIG); base(ax, relleno=True)
m = ax.scatter(e.geometry.x, e.geometry.y, s=28, c=e.ppt_anual, cmap="YlGnBu", vmin=0, vmax=4000,
               edgecolor="k", linewidth=0.4, zorder=3)
plantas.plot(ax=ax, color="#E0A526", marker="^", markersize=22, edgecolor="k", zorder=4)
barra(fig, ax, m, "Lluvia media anual en estaciones IDEAM (mm)"); guardar(fig, "07_estaciones_ideam.png")